# 🏗️ Vision Transformer Architecture Deep Dive

This notebook covers the detailed architecture of Vision Transformers with complete PyTorch implementations.

---

## 📚 **Learning Objectives**
- Understand each component of ViT architecture
- Implement patch embedding from scratch
- Build multi-head self-attention mechanism
- Create complete transformer encoder blocks
- Understand positional encoding strategies

---

## 🎯 **The ViT Pipeline**

```
Input Image → Patch Embedding → Positional Encoding → Transformer Encoder → Classification Head
   (H×W×C)      (N×D)              (N×D)               (N×D)                  (num_classes)
```

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import numpy as np
import matplotlib.pyplot as plt
from typing import Optional, Tuple, List
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🔧 Using device: {device}")

torch.manual_seed(42)
np.random.seed(42)

🔧 Using device: cuda


## 🔍 **1. Patch Embedding: From Pixels to Tokens**

### **What is Patch Embedding?**
Patch embedding is the first and most crucial step in Vision Transformers that bridges the gap between computer vision and natural language processing. While transformers were originally designed for sequences of discrete tokens (words), images are continuous 2D grids of pixels. Patch embedding solves this by:

1. **Dividing the image into non-overlapping patches** (like cutting a puzzle into pieces)
2. **Flattening each patch** into a 1D vector
3. **Linearly projecting** each patch to a fixed embedding dimension

### **Mathematical Formulation**
Given an input image **X ∈ ℝ^(H×W×C)**:
- **H, W**: Height and width of the image
- **C**: Number of channels (3 for RGB)
- **P**: Patch size (typically 16×16)

The patch embedding process creates **N = HW/P²** patches, where each patch becomes a token.

### **Why This Approach?**

**🎯 Advantages:**
- **Computational Efficiency**: Reduces sequence length from H×W pixels to N patches
- **Translation Invariance**: Each patch is processed identically
- **Scalability**: Works with different image sizes (with interpolation)

**🔄 Alternative Approaches:**
- **Pixel-level**: Too computationally expensive (224×224 = 50K tokens!)
- **Hierarchical**: More complex but used in some variants (PiT, TNT)

### **Implementation Details**

The patch embedding uses a **convolution with kernel_size = stride = patch_size**. This is mathematically equivalent to:
1. Splitting image into patches
2. Flattening each patch  
3. Applying linear projection

**Key Design Choices:**
- **Patch Size**: Smaller patches = more detail but more computation
- **Embedding Dimension**: Usually 768 (ViT-Base) or 1024 (ViT-Large)
- **Normalization**: Layer normalization helps training stability


In [3]:
class PatchEmbedding(nn.Module):
    """Convert image patches to embeddings.

    Args:
        img_size: Input image size
        patch_size: Size of each patch
        in_chans: Number of input channels
        embed_dim: Embedding dimension
    """

    def __init__(self, img_size: int = 224, patch_size: int = 16,
                 in_chans: int = 3, embed_dim: int = 768):
        super().__init__()
        self.img_size = img_size
        self.patch_size = patch_size
        self.n_patches = (img_size // patch_size) ** 2

        # Conv2d with kernel=patch_size, stride=patch_size
        # This is equivalent to splitting into patches + linear projection
        self.proj = nn.Conv2d(
            in_chans, embed_dim,
            kernel_size=patch_size,
            stride=patch_size
        )

        # Layer normalization
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Forward pass.

        Args:
            x: Input tensor of shape (B, C, H, W)

        Returns:
            Patch embeddings of shape (B, N, embed_dim)
        """
        B, C, H, W = x.shape
        assert H == self.img_size and W == self.img_size, \
            f"Input size ({H}x{W}) doesn't match expected size ({self.img_size}x{self.img_size})"

        # Patch embedding: (B, C, H, W) -> (B, embed_dim, H/P, W/P)
        x = self.proj(x)

        # Flatten spatial dimensions: (B, embed_dim, H/P, W/P) -> (B, embed_dim, N)
        x = x.flatten(2)

        # Transpose: (B, embed_dim, N) -> (B, N, embed_dim)
        x = x.transpose(1, 2)

        # Layer normalization
        x = self.norm(x)

        return x

# Test patch embedding
patch_embed = PatchEmbedding(img_size=224, patch_size=16, embed_dim=768)
sample_image = torch.randn(2, 3, 224, 224)  # Batch of 2 images
embedded_patches = patch_embed(sample_image)

print(f"🔍 Patch Embedding Demo:")
print(f"Input shape: {sample_image.shape}")
print(f"Output shape: {embedded_patches.shape}")
print(f"Number of patches: {patch_embed.n_patches}")
print(f"Embedding dimension: {embedded_patches.shape[-1]}")

🔍 Patch Embedding Demo:
Input shape: torch.Size([2, 3, 224, 224])
Output shape: torch.Size([2, 196, 768])
Number of patches: 196
Embedding dimension: 768


## 🎯 **2. The Special [CLS] Token**

### **What is the [CLS] Token?**
The [CLS] (classification) token is a **learnable embedding** that serves as an aggregate representation of the entire image. Borrowed from BERT in NLP, this special token is:

1. **Prepended** to the sequence of patch embeddings
2. **Randomly initialized** but learned during training
3. **Used for classification** - its final representation contains global image information

### **Why Do We Need [CLS]?**

**🤔 The Problem**: Unlike RNNs or CNNs, transformers process all tokens in parallel. There's no natural "final state" to use for classification.

**💡 The Solution**: Add a special token that can:
- **Attend to all patches** through self-attention
- **Aggregate global information** from the entire image
- **Serve as a summary** for downstream tasks

### **How Does [CLS] Work?**

**Step-by-Step Process:**
1. **Initialization**: [CLS] starts with random values
2. **Attention**: Through self-attention, [CLS] learns to focus on relevant patches
3. **Aggregation**: [CLS] accumulates information from all patches across layers
4. **Classification**: Final [CLS] representation goes to the classification head

**Mathematical Perspective:**
- Input: **[CLS], patch₁, patch₂, ..., patchₙ]**
- After L layers: **[CLS]ₗ** contains global image representation
- Classification: **logits = MLP([CLS]ₗ)**

### **Alternative Approaches**

**🔄 Other Aggregation Methods:**
- **Global Average Pooling**: Average all patch representations
- **Attention Pooling**: Weighted average based on learned attention
- **Multiple Tokens**: Use several special tokens (less common)

**🎯 Why [CLS] is Preferred:**
- **Learnable**: Adapts to specific tasks and datasets
- **Flexible**: Can focus on different regions as needed
- **Proven**: Works well empirically across many vision tasks

### **Training Dynamics**
- **Early Training**: [CLS] attends somewhat uniformly to all patches
- **Late Training**: [CLS] learns to focus on task-relevant regions
- **Specialization**: Different attention heads in [CLS] may focus on different semantic aspects


In [5]:
class CLSToken(nn.Module):
    """Learnable [CLS] token for classification.

    Args:
        embed_dim: Embedding dimension
    """

    def __init__(self, embed_dim: int = 768):
        super().__init__()
        # Learnable [CLS] token
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))

        # Initialize with small random values
        nn.init.trunc_normal_(self.cls_token, std=0.02)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Prepend [CLS] token to patch embeddings.

        Args:
            x: Patch embeddings of shape (B, N, embed_dim)

        Returns:
            Embeddings with [CLS] token of shape (B, N+1, embed_dim)
        """
        B = x.shape[0]

        # Expand [CLS] token for batch
        cls_tokens = self.cls_token.expand(B, -1, -1)

        # Concatenate [CLS] token with patch embeddings
        x = torch.cat([cls_tokens, x], dim=1)

        return x

# Test [CLS] token
cls_token = CLSToken(embed_dim=768)
embedded_with_cls = cls_token(embedded_patches)

print(f"🎯 [CLS] Token Demo:")
print(f"Before [CLS]: {embedded_patches.shape}")
print(f"After [CLS]: {embedded_with_cls.shape}")
print(f"[CLS] token shape: {cls_token.cls_token.shape}")

🎯 [CLS] Token Demo:
Before [CLS]: torch.Size([2, 196, 768])
After [CLS]: torch.Size([2, 197, 768])
[CLS] token shape: torch.Size([1, 1, 768])


## 📍 **3. Positional Encoding**

### **The Position Problem**
Transformers are **permutation invariant** - they process sequences without inherent order awareness. For text, word order matters ("cat chases mouse" ≠ "mouse chases cat"). For images, spatial relationships are equally crucial (edges, textures, objects have spatial structure).

### **Why Images Need Positional Information**

**🖼️ Spatial Relationships Matter:**
- **Object boundaries**: Adjacent patches often belong to the same object
- **Texture patterns**: Spatial continuity helps recognize textures
- **Geometric structure**: Shapes and arrangements have spatial meaning
- **Context**: Nearby patches provide contextual information

**⚠️ Without Positional Encoding:**
The model would treat patch sequences like "bag of patches" - losing all spatial structure!

### **Types of Positional Encoding**

#### **1. Learnable Positional Embedding (ViT Default)**
```python
pos_embed = nn.Parameter(torch.zeros(1, N+1, embed_dim))
```

**🎯 Advantages:**
- **Adaptive**: Learns optimal positional representation for each task
- **Flexible**: Can capture complex spatial relationships
- **Simple**: Easy to implement and understand

**📊 How it Works:**
- Each position gets a unique learnable vector
- Added element-wise to patch embeddings
- Trained end-to-end with the rest of the model

#### **2. Fixed Sinusoidal Encoding (From Original Transformer)**
**Mathematical Formula:**
```
PE(pos, 2i) = sin(pos / 10000^(2i/d_model))
PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))
```

**🔄 Advantages:**
- **No extra parameters**: Doesn't increase model size
- **Extrapolation**: Can handle longer sequences than seen during training
- **Interpretable**: Clear mathematical relationship

### **2D Positional Encoding for Images**

Images are 2D, but transformers expect 1D sequences. How do we encode 2D spatial relationships?

#### **Approach 1: Flattened 1D Encoding (ViT Default)**
```
Position mapping: (row, col) → row * width + col
```
- Treats spatial grid as flattened sequence
- Simple but loses explicit 2D structure

#### **Approach 2: Separate 2D Encoding**
```python
pos_embed_x = get_1d_encoding(width)  # Horizontal positions
pos_embed_y = get_1d_encoding(height) # Vertical positions
pos_embed_2d = pos_embed_x + pos_embed_y  # Combine
```

#### **Approach 3: Learned 2D Factorized**
- Learn separate embeddings for row and column
- More explicitly captures 2D nature

### **Position Interpolation**

**🔍 The Challenge**: What if we want to use different image sizes than training?

**💡 The Solution**: **Positional Interpolation**
```python
# If trained on 224x224 but testing on 384x384
old_pos_embed = model.pos_embed  # Shape: (1, 197, 768)
new_pos_embed = interpolate_pos_encoding(old_pos_embed, new_size)
```

**How it Works:**
1. **Exclude [CLS]**: Keep [CLS] position unchanged
2. **Reshape**: Convert 1D positions back to 2D grid
3. **Interpolate**: Use bicubic interpolation to new size
4. **Flatten**: Convert back to 1D sequence

### **Training Considerations**

**🎯 Initialization Strategy:**
- **Learnable**: Usually initialized to zeros or small random values
- **Fixed**: Use mathematical formula (sinusoidal)

**📈 Training Dynamics:**
- **Early Training**: Positional embeddings learn basic spatial structure
- **Mid Training**: Develops understanding of spatial relationships
- **Late Training**: Fine-tunes for task-specific spatial patterns

**🔧 Practical Tips:**
- **Dropout**: Usually not applied to positional embeddings
- **Sharing**: Same positional embedding used across all layers
- **Scale**: Sometimes scaled by √(embed_dim) for stability


In [6]:
class PositionalEncoding(nn.Module):
    """Positional encoding for ViT.

    Args:
        n_patches: Number of patches
        embed_dim: Embedding dimension
        encoding_type: 'learnable' or 'sinusoidal'
    """

    def __init__(self, n_patches: int, embed_dim: int,
                 encoding_type: str = 'learnable'):
        super().__init__()
        self.n_patches = n_patches
        self.embed_dim = embed_dim
        self.encoding_type = encoding_type

        if encoding_type == 'learnable':
            # Learnable positional embeddings (ViT default)
            self.pos_embed = nn.Parameter(
                torch.zeros(1, n_patches + 1, embed_dim)  # +1 for [CLS] token
            )
            nn.init.trunc_normal_(self.pos_embed, std=0.02)

        elif encoding_type == 'sinusoidal':
            # Fixed sinusoidal positional encoding
            self.register_buffer(
                'pos_embed',
                self._get_sinusoidal_encoding(n_patches + 1, embed_dim)
            )

    def _get_sinusoidal_encoding(self, n_positions: int, embed_dim: int) -> torch.Tensor:
        """Generate sinusoidal positional encodings.

        Args:
            n_positions: Number of positions
            embed_dim: Embedding dimension

        Returns:
            Positional encodings of shape (1, n_positions, embed_dim)
        """
        position = torch.arange(n_positions).unsqueeze(1).float()
        div_term = torch.exp(
            torch.arange(0, embed_dim, 2).float() *
            -(math.log(10000.0) / embed_dim)
        )

        pos_encoding = torch.zeros(n_positions, embed_dim)
        pos_encoding[:, 0::2] = torch.sin(position * div_term)
        pos_encoding[:, 1::2] = torch.cos(position * div_term)

        return pos_encoding.unsqueeze(0)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Add positional encoding to embeddings.

        Args:
            x: Input embeddings of shape (B, N+1, embed_dim)

        Returns:
            Embeddings with positional encoding
        """
        return x + self.pos_embed

# Test positional encoding
pos_encoding = PositionalEncoding(n_patches=196, embed_dim=768, encoding_type='learnable')
embedded_with_pos = pos_encoding(embedded_with_cls)

print(f"📍 Positional Encoding Demo:")
print(f"Input shape: {embedded_with_cls.shape}")
print(f"Output shape: {embedded_with_pos.shape}")
print(f"Position embedding shape: {pos_encoding.pos_embed.shape}")

📍 Positional Encoding Demo:
Input shape: torch.Size([2, 197, 768])
Output shape: torch.Size([2, 197, 768])
Position embedding shape: torch.Size([1, 197, 768])


## 🧠 **4. Multi-Head Self-Attention (MHSA)**

### **The Heart of Transformers**
Multi-Head Self-Attention is the **core mechanism** that makes transformers so powerful. It allows every patch to "look at" and gather information from every other patch in the image, enabling global context understanding from the very first layer.

### **What is Self-Attention?**

**🎯 Core Concept**: Each token (patch) computes how much it should "attend to" every other token, including itself.

**💭 Intuitive Understanding**:
- Imagine you're looking at a photo of a cat
- The patch containing the cat's eye might attend strongly to patches containing whiskers, ears, and nose
- The background patches might attend to each other to understand scene context
- Every patch can dynamically focus on the most relevant parts of the image

### **Mathematical Foundation**

#### **Step 1: Create Queries, Keys, and Values**
For each token embedding **x**, we create three vectors:
- **Query (Q)**: "What am I looking for?"
- **Key (K)**: "What information do I have?"
- **Value (V)**: "What information do I provide?"

```
Q = x × W_Q    (What the token is searching for)
K = x × W_K    (What the token offers as context)
V = x × W_V    (What the token contributes)
```

#### **Step 2: Compute Attention Scores**
```
Attention_Scores = (Q × K^T) / √(d_k)
```
- **Q × K^T**: Dot product measures similarity between query and keys
- **√(d_k)**: Scaling factor to prevent vanishing gradients

#### **Step 3: Apply Softmax**
```
Attention_Weights = softmax(Attention_Scores)
```
- Converts scores to probabilities (sum = 1)
- Higher scores → higher attention weights

#### **Step 4: Weighted Sum of Values**
```
Output = Attention_Weights × V
```
- Each token gets updated representation based on attended information

### **Why Multiple Heads?**

**🤔 Single Head Limitation**: One attention head can only capture one type of relationship.

**💡 Multi-Head Solution**: Different heads can specialize in different aspects:
- **Head 1**: Might focus on spatial proximity (neighboring patches)
- **Head 2**: Might focus on color similarity
- **Head 3**: Might focus on texture patterns
- **Head 4**: Might focus on object boundaries

**📊 Implementation**:
```
head_dim = embed_dim // num_heads
Each head operates on smaller dimension (64 for ViT-Base: 768/12)
```

### **Attention Patterns in Vision**

#### **Spatial Attention Patterns**
- **Local Patterns**: Some heads focus on neighboring patches (like convolution)
- **Global Patterns**: Other heads look at distant but semantically related patches
- **Object-Centric**: Heads might attend within object boundaries

#### **Semantic Attention Patterns**
- **Texture**: Patches with similar textures attend to each other
- **Color**: Similar colored regions show high attention
- **Shape**: Parts of the same object boundary attend strongly

### **Comparison with CNNs**

| Aspect | CNN | Self-Attention |
|--------|-----|----------------|
| **Receptive Field** | Grows gradually | Global from layer 1 |
| **Computation** | Fixed convolution | Dynamic, content-dependent |
| **Inductive Bias** | Strong spatial bias | Minimal inductive bias |
| **Flexibility** | Less adaptive | Highly adaptive |
| **Efficiency** | O(1) per location | O(N²) for N patches |

### **Computational Complexity**

For **N patches** and **D embedding dimension**:
- **QKV Linear Projections**: O(N × D²)
- **Attention Matrix**: O(N² × D)
- **Memory**: O(N²) for attention weights

**⚠️ Quadratic Scaling**: Main limitation - attention matrix grows quadratically with sequence length.

### **Advanced Attention Mechanisms**

#### **1. Scaled Dot-Product Attention (Used in ViT)**
```
Attention(Q,K,V) = softmax(QK^T/√d_k)V
```

#### **2. Window-Based Attention (Swin Transformer)**
- Restricts attention to local windows
- Reduces complexity from O(N²) to O(N)

#### **3. Linear Attention Variants**
- Various approximations to reduce quadratic complexity
- Trade-off between efficiency and performance

### **Training Dynamics of Attention**

**🚀 Early Training**:
- Attention patterns are somewhat random
- Gradual development of spatial awareness

**📈 Mid Training**:
- Clear spatial patterns emerge
- Different heads develop distinct specializations

**🎯 Late Training**:
- Fine-tuned, task-specific attention patterns
- [CLS] token learns to attend to relevant regions

### **Visualization and Interpretation**

**📊 What Can We Learn from Attention Maps?**
- **Model Focus**: Which parts of the image are important?
- **Spatial Relationships**: How do patches relate to each other?
- **Feature Extraction**: What visual patterns are captured?
- **Failure Modes**: Where does the model struggle?

**🔍 Practical Uses**:
- **Debugging**: Understanding model decisions
- **Explainability**: Showing which regions influence predictions
- **Analysis**: Comparing different model architectures


In [7]:
class MultiHeadSelfAttention(nn.Module):
    """Multi-Head Self-Attention mechanism.

    Args:
        embed_dim: Embedding dimension
        num_heads: Number of attention heads
        dropout: Dropout probability
    """

    def __init__(self, embed_dim: int = 768, num_heads: int = 12,
                 dropout: float = 0.1):
        super().__init__()
        assert embed_dim % num_heads == 0, "embed_dim must be divisible by num_heads"

        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.scale = self.head_dim ** -0.5  # Scaling factor for attention

        # Linear projections for Q, K, V
        self.qkv = nn.Linear(embed_dim, embed_dim * 3, bias=False)

        # Output projection
        self.proj = nn.Linear(embed_dim, embed_dim)

        # Dropout
        self.attn_dropout = nn.Dropout(dropout)
        self.proj_dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor,
                return_attention: bool = False) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
        """Forward pass.

        Args:
            x: Input tensor of shape (B, N, embed_dim)
            return_attention: Whether to return attention weights

        Returns:
            Output tensor and optionally attention weights
        """
        B, N, C = x.shape

        # Generate Q, K, V
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)  # (3, B, num_heads, N, head_dim)
        q, k, v = qkv[0], qkv[1], qkv[2]  # Each: (B, num_heads, N, head_dim)

        # Compute attention scores
        attn = (q @ k.transpose(-2, -1)) * self.scale  # (B, num_heads, N, N)
        attn = attn.softmax(dim=-1)
        attn = self.attn_dropout(attn)

        # Apply attention to values
        x = (attn @ v).transpose(1, 2).reshape(B, N, C)  # (B, N, embed_dim)

        # Output projection
        x = self.proj(x)
        x = self.proj_dropout(x)

        if return_attention:
            return x, attn
        return x, None

# Test multi-head self-attention
mhsa = MultiHeadSelfAttention(embed_dim=768, num_heads=12)
attn_output, attn_weights = mhsa(embedded_with_pos, return_attention=True)

print(f"🧠 Multi-Head Self-Attention Demo:")
print(f"Input shape: {embedded_with_pos.shape}")
print(f"Output shape: {attn_output.shape}")
print(f"Attention weights shape: {attn_weights.shape}")
print(f"Number of heads: {mhsa.num_heads}")
print(f"Head dimension: {mhsa.head_dim}")

🧠 Multi-Head Self-Attention Demo:
Input shape: torch.Size([2, 197, 768])
Output shape: torch.Size([2, 197, 768])
Attention weights shape: torch.Size([2, 12, 197, 197])
Number of heads: 12
Head dimension: 64


## 🔗 **5. Feed-Forward Network (MLP)**

### **The Processing Powerhouse**
While self-attention handles **inter-token relationships**, the MLP (Multi-Layer Perceptron) provides the **individual token processing power**. Think of it as the "thinking" component that processes the information gathered by attention.

### **What Does the MLP Do?**

**🧠 Core Function**: The MLP applies **position-wise transformations** to each token independently:
- **Input**: Token representations after attention
- **Process**: Non-linear transformations to extract features
- **Output**: Enhanced token representations

**🎯 Key Responsibilities**:
1. **Feature Extraction**: Discovers complex patterns within token representations
2. **Non-linearity**: Introduces crucial non-linear transformations
3. **Capacity**: Provides most of the model's learnable parameters
4. **Specialization**: Learns task-specific transformations

### **Architecture Details**

#### **Standard ViT MLP Structure**:
```
Input (768) → Linear (3072) → GELU → Dropout → Linear (768) → Dropout → Output (768)
```

**📊 Expansion-Contraction Pattern**:
- **Expansion**: `embed_dim → mlp_ratio × embed_dim` (typically 4x)
- **Contraction**: `mlp_ratio × embed_dim → embed_dim`
- **Purpose**: Creates a bottleneck that forces feature compression and extraction

### **Why This Design?**

#### **1. Computational Trade-offs**
- **Wide Hidden Layer**: More capacity for complex transformations
- **4x Expansion**: Empirically found to work well (inherited from NLP transformers)
- **Bottleneck**: Forces the model to learn compressed, useful representations

#### **2. Universal Approximation**
- **Theoretical Foundation**: MLPs with sufficient width can approximate any continuous function
- **Practical Benefit**: Can learn complex, non-linear relationships in the data

### **Activation Functions: Why GELU?**

#### **GELU (Gaussian Error Linear Unit)**
```python
GELU(x) = x × Φ(x) = x × (1/2)[1 + erf(x/√2)]
```

**🎯 Advantages over ReLU**:
- **Smooth**: Differentiable everywhere (better gradients)
- **Probabilistic**: Has a probabilistic interpretation
- **Performance**: Often works better in transformers

**📊 Comparison of Activations**:

| Activation | Formula | Pros | Cons |
|------------|---------|------|------|
| **ReLU** | max(0, x) | Simple, fast | Dead neurons, non-smooth |
| **GELU** | x × Φ(x) | Smooth, probabilistic | Slightly more computation |
| **Swish** | x × sigmoid(x) | Smooth, unbounded above | More complex |


### **Role in the Transformer Block**

#### **Information Flow**:
```
Input → LayerNorm → Self-Attention → Residual Connection
                                    ↓
Output ← LayerNorm ← MLP ← Residual Connection
```

**🔄 Why After Attention?**:
1. **Attention gathers information** from other tokens
2. **MLP processes this aggregated information** individually per token
3. **Creates a two-stage processing pipeline**: gather → process

### **Parameter Distribution**

For a ViT-Base model (embed_dim=768, mlp_ratio=4):
- **MLP Parameters per layer**: ~4.7M parameters
- **Attention Parameters per layer**: ~2.4M parameters
- **MLP dominates**: ~66% of transformer block parameters

**📈 Scaling Implications**:
- **Larger MLPs**: More capacity but also more overfitting risk
- **Smaller MLPs**: More efficient but potentially underpowered
- **Sweet Spot**: 4x expansion works well across many tasks

### **Training Dynamics**

#### **Early Training**:
- **Random Transformations**: MLPs start with random non-linear mappings
- **Gradient Flow**: Residual connections help gradients flow

#### **Mid Training**:
- **Feature Specialization**: Different neurons start specializing
- **Pattern Recognition**: MLPs learn to recognize visual patterns

#### **Late Training**:
- **Task Optimization**: Fine-tune for specific classification tasks
- **Overparameterization**: Many neurons may become redundant

### **Variants and Improvements**

#### **1. Different Expansion Ratios**
- **2x**: More efficient, slightly lower performance
- **8x**: More capacity, risk of overfitting
- **Adaptive**: Different ratios per layer

#### **2. Alternative Architectures**
- **ConvFFN**: Replace linear layers with convolutions
- **MoE (Mixture of Experts)**: Conditional computation with expert routing
- **GLU Variants**: Gated Linear Units for better information flow

#### **3. Efficiency Optimizations**
- **Linear Attention**: Replace some MLP capacity with efficient attention
- **Structured Pruning**: Remove less important neurons
- **Knowledge Distillation**: Train smaller MLPs to mimic larger ones

### **Relationship with CNNs**

**🔄 Functional Similarity**:
- **CNN**: Fixed local transformations (convolution + activation)
- **MLP**: Learned global transformations (linear + activation)

**📊 Key Differences**:
| Aspect | CNN | MLP in ViT |
|--------|-----|------------|
| **Scope** | Local neighborhood | Individual token |
| **Parameters** | Shared across positions | Position-independent |
| **Inductive Bias** | Translation equivariance | Minimal bias |
| **Capacity** | Fixed by kernel size | Flexible by width |

### **Practical Considerations**

#### **Memory Usage**:
- **Activation Memory**: Dominant during training (4x expansion)
- **Gradient Memory**: Significant during backpropagation
- **Optimization**: Use gradient checkpointing for large models

#### **Computational Efficiency**:
- **Matrix Multiplication**: Highly optimized on modern hardware
- **Parallelization**: Each token processed independently
- **Batch Processing**: Efficient batched linear operations

#### **Hyperparameter Sensitivity**:
- **MLP Ratio**: Usually 4, but worth experimenting
- **Dropout Rate**: Critical for preventing overfitting
- **Initialization**: Important for training stability


In [8]:
class MLP(nn.Module):
    """Multi-Layer Perceptron (Feed-Forward Network).

    Args:
        embed_dim: Input embedding dimension
        mlp_ratio: Ratio of hidden dimension to embedding dimension
        dropout: Dropout probability
    """

    def __init__(self, embed_dim: int = 768, mlp_ratio: float = 4.0,
                 dropout: float = 0.1):
        super().__init__()
        hidden_dim = int(embed_dim * mlp_ratio)

        self.fc1 = nn.Linear(embed_dim, hidden_dim)
        self.act = nn.GELU()  # GELU activation
        self.fc2 = nn.Linear(hidden_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Forward pass.

        Args:
            x: Input tensor of shape (B, N, embed_dim)

        Returns:
            Output tensor of shape (B, N, embed_dim)
        """
        x = self.fc1(x)
        x = self.act(x)
        x = self.dropout(x)
        x = self.fc2(x)
        x = self.dropout(x)
        return x

# Test MLP
mlp = MLP(embed_dim=768, mlp_ratio=4.0)
mlp_output = mlp(attn_output)

print(f"🔗 MLP Demo:")
print(f"Input shape: {attn_output.shape}")
print(f"Output shape: {mlp_output.shape}")
print(f"Hidden dimension: {int(768 * 4.0)}")

🔗 MLP Demo:
Input shape: torch.Size([2, 197, 768])
Output shape: torch.Size([2, 197, 768])
Hidden dimension: 3072


## 🏗️ **6. Transformer Encoder Block**

### **The Building Block of ViT**
The Transformer Block is the **fundamental unit** that gets stacked to create the complete ViT architecture. Each block combines the power of self-attention and feed-forward processing with crucial architectural innovations that enable deep, stable training.

### **Architecture Overview**

```
Input Token Embeddings (B, N, D)
        ↓
    LayerNorm
        ↓
Multi-Head Self-Attention  ←─ Skip Connection (Residual)
        ↓                      ↓
        ├──────────────────────┘
        ↓
    LayerNorm
        ↓
Feed-Forward Network (MLP) ←─ Skip Connection (Residual)
        ↓                      ↓
        ├──────────────────────┘
        ↓
Output Token Embeddings (B, N, D)
```

### **Key Design Principles**

#### **1. Pre-Layer Normalization (Pre-Norm)**
**ViT uses Pre-Norm architecture**:
```python
# Pre-Norm (ViT style)
x = x + attention(layer_norm(x))
x = x + mlp(layer_norm(x))
```

**vs. Post-Norm (Original Transformer)**:
```python
# Post-Norm (Original Transformer style)
x = layer_norm(x + attention(x))
x = layer_norm(x + mlp(x))
```

**🎯 Why Pre-Norm?**
- **Training Stability**: Easier to train deep networks
- **Gradient Flow**: Better gradient flow through residual connections
- **Normalization**: Input to each sub-layer is normalized

#### **2. Residual Connections (Skip Connections)**
**Purpose**: Enable information to flow directly from input to output, bypassing the transformation layers.

**Mathematical Representation**:
```
H₁ = x + MultiHeadAttention(LayerNorm(x))
H₂ = H₁ + MLP(LayerNorm(H₁))
```

**🚀 Benefits**:
- **Gradient Flow**: Prevents vanishing gradient problem
- **Identity Mapping**: Model can learn to ignore layers if needed
- **Training Stability**: Enables training of very deep networks
- **Feature Preservation**: Maintains low-level features alongside high-level ones

### **Layer Normalization Deep Dive**

#### **What is Layer Normalization?**
Normalizes each feature across the embedding dimension for each token independently.

**Formula**:
```
LayerNorm(x) = γ × (x - μ) / σ + β
```
Where:
- **μ**: Mean across embedding dimension
- **σ**: Standard deviation across embedding dimension  
- **γ, β**: Learnable scale and shift parameters

#### **Why Not Batch Normalization?**
| Aspect | Batch Norm | Layer Norm |
|--------|------------|------------|
| **Normalization Axis** | Across batch | Across features |
| **Sequence Length** | Sensitive to varying lengths | Length invariant |
| **Inference** | Requires running statistics | No external dependencies |
| **Transformers** | Problematic | Perfect fit |

### **Information Flow Analysis**

#### **Within a Single Block**:
1. **Input Preparation**: Layer normalization prepares input for attention
2. **Global Context**: Self-attention gathers information from all patches
3. **Residual Addition**: Combines new information with original input
4. **Feature Processing**: Second layer norm + MLP processes the enhanced features
5. **Final Combination**: Second residual connection preserves information

#### **Across Multiple Blocks**:
- **Progressive Refinement**: Each block refines the representations
- **Hierarchical Features**: Early blocks capture low-level features, later blocks capture high-level semantics
- **Global-to-Specific**: Attention patterns often go from broad to focused

### **Training Dynamics**

#### **Gradient Flow Properties**:
**Without Residual Connections**:
```
∂Loss/∂x = ∂Loss/∂output × ∂output/∂attention × ∂attention/∂x
```
- Gradients can vanish or explode through multiple multiplications

**With Residual Connections**:
```
∂Loss/∂x = ∂Loss/∂output × (1 + ∂residual_path/∂x)
```
- Always has a direct path with gradient = 1

#### **Layer-wise Learning Rates**:
Due to residual connections, different layers can have different effective learning rates:
- **Early Layers**: Often learn slower (closer to input)
- **Later Layers**: Often learn faster (closer to loss)
- **Adaptive Optimizers**: Help balance this naturally

### **Computational Efficiency**

#### **Memory Complexity**:
For **B** batch size, **N** sequence length, **D** embedding dimension:
- **Attention Matrix**: O(B × H × N²) where H is number of heads
- **Activations**: O(B × N × D) per layer
- **Gradients**: Similar to activations during backprop

#### **Time Complexity**:
- **Self-Attention**: O(N² × D) - quadratic in sequence length
- **MLP**: O(N × D²) - linear in sequence length
- **Total per Block**: Dominated by attention for long sequences

### **Stacking Multiple Blocks**

#### **Depth vs. Width Trade-offs**:
- **Deeper Networks**: More layers, more complex hierarchical features
- **Wider Networks**: Larger embedding dimensions, more representational capacity

#### **Typical Configurations**:
| Model | Layers | Embed Dim | Heads | Parameters |
|-------|--------|-----------|-------|------------|
| **ViT-Small** | 12 | 384 | 6 | 22M |
| **ViT-Base** | 12 | 768 | 12 | 86M |
| **ViT-Large** | 24 | 1024 | 16 | 307M |
| **ViT-Huge** | 32 | 1280 | 16 | 632M |

### **Attention Pattern Evolution**

#### **Early Layers (1-4)**:
- **Local Patterns**: Often resemble convolution-like patterns
- **Spatial Proximity**: High attention between neighboring patches
- **Low-level Features**: Edge detection, color patterns

#### **Middle Layers (5-8)**:
- **Object Parts**: Attention within object boundaries
- **Texture Patterns**: Similar textures attend to each other
- **Semantic Grouping**: Related visual elements cluster

#### **Late Layers (9-12)**:
- **Global Context**: Long-range dependencies
- **Task-Specific**: Attention patterns specific to classification task
- **[CLS] Specialization**: [CLS] token attends to task-relevant regions

### **Common Failure Modes and Solutions**

#### **1. Training Instability**:
**Problem**: Large gradients, oscillating loss
**Solution**:
- Careful initialization (Gaussian with std=0.02)
- Gradient clipping
- Lower learning rate

#### **2. Attention Collapse**:
**Problem**: All attention heads learn similar patterns
**Solution**:
- Different initialization for different heads
- Attention dropout
- Regularization techniques

#### **3. Overfitting**:
**Problem**: Good training performance, poor validation
**Solution**:
- Dropout in MLP and attention
- Data augmentation
- Pre-training on larger datasets

### **Variants and Modifications**

#### **1. Post-Layer Norm**:
Some variants use post-norm for different training characteristics

#### **2. RMSNorm**:
Alternative normalization that doesn't center the data:
```python
RMSNorm(x) = x / RMS(x) × γ
```

#### **3. Different Activation Functions**:
- **SwiGLU**: Swish with GLU gating
- **GeGLU**: GELU with GLU gating
- **ReLU variants**: For computational efficiency

#### **4. Efficient Attention**:
- **Linear Attention**: Reduces quadratic complexity
- **Sparse Attention**: Only attend to subset of positions
- **Local Attention**: Restrict attention to local windows


In [9]:
class TransformerBlock(nn.Module):
    """Transformer encoder block.

    Args:
        embed_dim: Embedding dimension
        num_heads: Number of attention heads
        mlp_ratio: MLP expansion ratio
        dropout: Dropout probability
    """

    def __init__(self, embed_dim: int = 768, num_heads: int = 12,
                 mlp_ratio: float = 4.0, dropout: float = 0.1):
        super().__init__()

        # Layer normalization (Pre-norm architecture)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)

        # Multi-head self-attention
        self.attn = MultiHeadSelfAttention(embed_dim, num_heads, dropout)

        # Feed-forward network
        self.mlp = MLP(embed_dim, mlp_ratio, dropout)

    def forward(self, x: torch.Tensor,
                return_attention: bool = False) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
        """Forward pass with residual connections.

        Args:
            x: Input tensor of shape (B, N, embed_dim)
            return_attention: Whether to return attention weights

        Returns:
            Output tensor and optionally attention weights
        """
        # Self-attention with residual connection (Pre-norm)
        attn_out, attn_weights = self.attn(self.norm1(x), return_attention)
        x = x + attn_out  # Residual connection

        # MLP with residual connection (Pre-norm)
        mlp_out = self.mlp(self.norm2(x))
        x = x + mlp_out  # Residual connection

        return x, attn_weights

# Test transformer block
transformer_block = TransformerBlock(embed_dim=768, num_heads=12)
block_output, block_attn = transformer_block(embedded_with_pos, return_attention=True)

print(f"🏗️ Transformer Block Demo:")
print(f"Input shape: {embedded_with_pos.shape}")
print(f"Output shape: {block_output.shape}")
print(f"Attention shape: {block_attn.shape}")

🏗️ Transformer Block Demo:
Input shape: torch.Size([2, 197, 768])
Output shape: torch.Size([2, 197, 768])
Attention shape: torch.Size([2, 12, 197, 197])


## 🎯 **7. Complete Vision Transformer**

### **Bringing It All Together**
The complete Vision Transformer combines all the components we've explored into a unified architecture that can learn powerful visual representations from data. Let's understand how all pieces work together and the design decisions that make ViT successful.

### **End-to-End Architecture Flow**

```
Raw Image (224×224×3)
        ↓
🔍 Patch Embedding (196 patches of 16×16)
        ↓
🎯 [CLS] Token Addition (197 tokens total)
        ↓  
📍 Positional Encoding (learnable positions)
        ↓
🏗️ Transformer Encoder Blocks ×12
   ├─ Multi-Head Self-Attention
   ├─ Residual Connection
   ├─ Layer Normalization  
   ├─ Feed-Forward Network
   └─ Residual Connection
        ↓
🧠 Final Layer Normalization
        ↓
🎯 [CLS] Token Extraction
        ↓
📊 Classification Head (Linear Layer)
        ↓
🏷️ Class Logits (1000 classes)
```

### **Model Variants and Scaling**

#### **Standard ViT Configurations**:

| Variant | Layers | Hidden Size | MLP Size | Heads | Params |
|---------|--------|-------------|----------|-------|--------|
| **ViT-Tiny** | 12 | 192 | 768 | 3 | 5.7M |
| **ViT-Small** | 12 | 384 | 1536 | 6 | 22M |
| **ViT-Base** | 12 | 768 | 3072 | 12 | 86M |
| **ViT-Large** | 24 | 1024 | 4096 | 16 | 307M |
| **ViT-Huge** | 32 | 1280 | 5120 | 16 | 632M |

#### **Patch Size Variants**:
- **ViT-Base/32**: 32×32 patches (49 tokens) - Less detail, faster
- **ViT-Base/16**: 16×16 patches (196 tokens) - Standard configuration  
- **ViT-Base/8**: 8×8 patches (784 tokens) - More detail, slower

### **Key Design Innovations**

#### **1. Minimal Inductive Biases**
**Philosophy**: Let the data teach the model about visual structure rather than hard-coding assumptions.

**Contrast with CNNs**:
- **CNNs**: Built-in translation equivariance, local connectivity
- **ViT**: Only patch embedding introduces spatial structure

**Benefits**:
- **Flexibility**: Can learn diverse visual patterns
- **Generalization**: Works across different visual domains
- **Scalability**: Benefits more from larger datasets

#### **2. Global Receptive Field from Layer 1**
**Revolutionary Change**: Every patch can attend to every other patch from the first layer.

**Implications**:
- **Long-range Dependencies**: Captures global context immediately
- **Object Relationships**: Understands spatial relationships across the image
- **Holistic Processing**: Considers entire image context for each decision

#### **3. Parameter Sharing Across Positions**
**Efficiency**: Same transformer weights process all spatial positions.

**Benefits**:
- **Parameter Efficiency**: Fewer parameters than position-specific processing
- **Translation Invariance**: Same processing regardless of patch position
- **Scalability**: Can handle different image sizes (with position interpolation)

### **Training Strategies**

#### **1. Large-Scale Pre-training**
**The ViT Recipe**:
```
Pre-training: Large dataset (ImageNet-21K, JFT-300M)
    ↓
Fine-tuning: Target dataset (ImageNet-1K, CIFAR, etc.)
```

**Why This Works**:
- **Data Hunger**: ViT needs more data than CNNs to overcome lack of inductive biases
- **Transfer Learning**: Pre-trained representations transfer well across tasks
- **Scaling Laws**: Performance improves predictably with more data and parameters

#### **2. Data Augmentation**
**Strong Augmentation is Crucial**:
- **MixUp**: Blend images and labels
- **CutMix**: Replace patches between images
- **RandAugment**: Automated augmentation policies
- **AutoAugment**: Learned augmentation strategies

#### **3. Regularization Techniques**
- **Dropout**: Applied in attention and MLP layers
- **Path Dropout**: Randomly skip entire layers during training
- **Label Smoothing**: Soften hard classification targets
- **Weight Decay**: L2 regularization on parameters

### **Comparison with CNNs**

#### **Architectural Differences**:

| Aspect | CNNs | Vision Transformers |
|--------|------|-------------------|
| **Receptive Field** | Local → Global | Global from start |
| **Inductive Bias** | Strong (locality, translation) | Minimal |
| **Parameter Sharing** | Within kernels | Across positions |
| **Data Requirements** | Moderate | High |
| **Computational Complexity** | O(N) | O(N²) |
| **Interpretability** | Feature maps | Attention maps |

#### **Performance Characteristics**:

**ViT Advantages**:
- **Large-scale Performance**: Scales better with more data
- **Transfer Learning**: Excellent cross-domain transfer
- **Long-range Dependencies**: Natural global context modeling
- **Simplicity**: Unified architecture across modalities

**CNN Advantages**:
- **Sample Efficiency**: Better with limited data
- **Computational Efficiency**: Linear complexity
- **Inductive Bias**: Built-in spatial understanding
- **Hardware Optimization**: Highly optimized implementations

### **Memory and Computational Analysis**

#### **Memory Complexity**:
For input size **H×W**, patch size **P**, embedding dimension **D**:

**Components**:
- **Patch Embeddings**: O(HW/P² × D)
- **Positional Embeddings**: O(HW/P² × D)  
- **Attention Matrices**: O(L × H × (HW/P²)²) where L=layers, H=heads
- **MLP Activations**: O(L × HW/P² × 4D)

**Scaling Issues**:
- **Quadratic in Image Size**: Attention complexity scales as O(N²)
- **Linear in Model Size**: MLP dominates parameter count
- **Memory Bottleneck**: Attention matrices for high-resolution images

#### **Computational Efficiency Improvements**:

**1. Efficient Attention Variants**:
- **Linear Attention**: Approximate attention with linear complexity
- **Sparse Attention**: Only attend to subset of positions
- **Window Attention**: Restrict attention to local windows (Swin Transformer)

**2. Model Compression**:
- **Knowledge Distillation**: Train smaller models to mimic larger ones
- **Pruning**: Remove less important parameters/heads
- **Quantization**: Reduce precision of weights and activations

### **Success Factors**

#### **1. Scale**
**Data Scale**: ViT truly shines with massive datasets
- **ImageNet-1K**: Good performance, but not exceptional
- **ImageNet-21K**: Competitive with best CNNs
- **JFT-300M**: State-of-the-art performance

**Model Scale**: Larger models generally perform better
- **Parameter Scaling**: More parameters → better performance
- **Compute Scaling**: More training compute → better performance

#### **2. Pre-training Strategy**
**Self-Supervised Learning**: Recent advances show promise
- **MAE (Masked Autoencoder)**: Mask patches, predict missing content
- **SimMIM**: Simple masked image modeling
- **BEiT**: BERT-style pre-training for images

#### **3. Architecture Innovations**
**Hybrid Approaches**:
- **ConViT**: Convolution-augmented ViT
- **CvT**: Convolutional Vision Transformer
- **CoaT**: Co-Scale Conv-Attentional Image Transformer

### **Limitations and Solutions**

#### **Current Limitations**:

**1. Computational Cost**:
- **Problem**: O(N²) attention complexity
- **Solutions**: Efficient attention, hierarchical processing

**2. Data Hunger**:
- **Problem**: Requires large datasets for good performance
- **Solutions**: Better pre-training, data augmentation, hybrid architectures

**3. Fine-grained Details**:
- **Problem**: May miss small visual details due to patch-based processing
- **Solutions**: Smaller patches, multi-scale processing

#### **Active Research Directions**:

**1. Efficiency**:
- **Mobile ViTs**: Efficient variants for deployment
- **Knowledge Distillation**: Compress large models
- **Neural Architecture Search**: Automated design optimization

**2. Multi-Modal Learning**:
- **CLIP**: Joint vision-language understanding
- **DALL-E**: Text-to-image generation
- **Flamingo**: Few-shot learning across modalities

**3. Self-Supervised Learning**:
- **MAE**: Masked autoencoding for vision
- **DINO**: Self-distillation for vision transformers
- **MoCo v3**: Momentum contrastive learning

### **Practical Implementation Tips**

#### **Training Considerations**:
- **Learning Rate**: Often requires lower LR than CNNs
- **Warmup**: Linear warmup crucial for training stability
- **Gradient Clipping**: Prevents exploding gradients
- **Mixed Precision**: Use FP16 for memory efficiency

#### **Hardware Requirements**:
- **Memory**: High memory requirements due to attention matrices
- **Compute**: Benefits from tensor cores (V100, A100)
- **Parallelization**: Excellent scalability across multiple GPUs

#### **Hyperparameter Sensitivity**:
- **Patch Size**: Trade-off between detail and efficiency
- **Dropout Rate**: Important for preventing overfitting
- **Attention Heads**: More heads generally better, but diminishing returns


In [10]:
class VisionTransformer(nn.Module):
    """Vision Transformer implementation.

    Args:
        img_size: Input image size
        patch_size: Patch size
        in_chans: Number of input channels
        num_classes: Number of output classes
        embed_dim: Embedding dimension
        depth: Number of transformer layers
        num_heads: Number of attention heads
        mlp_ratio: MLP expansion ratio
        dropout: Dropout probability
    """

    def __init__(self, img_size: int = 224, patch_size: int = 16,
                 in_chans: int = 3, num_classes: int = 1000,
                 embed_dim: int = 768, depth: int = 12,
                 num_heads: int = 12, mlp_ratio: float = 4.0,
                 dropout: float = 0.1):
        super().__init__()

        # Patch embedding
        self.patch_embed = PatchEmbedding(img_size, patch_size, in_chans, embed_dim)
        num_patches = self.patch_embed.n_patches

        # [CLS] token
        self.cls_token = CLSToken(embed_dim)

        # Positional embedding
        self.pos_embed = PositionalEncoding(num_patches, embed_dim)

        # Transformer encoder blocks
        self.blocks = nn.ModuleList([
            TransformerBlock(embed_dim, num_heads, mlp_ratio, dropout)
            for _ in range(depth)
        ])

        # Final layer norm
        self.norm = nn.LayerNorm(embed_dim)

        # Classification head
        self.head = nn.Linear(embed_dim, num_classes)

        # Initialize weights
        self.apply(self._init_weights)

    def _init_weights(self, m):
        """Initialize weights."""
        if isinstance(m, nn.Linear):
            nn.init.trunc_normal_(m.weight, std=0.02)
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.LayerNorm):
            nn.init.constant_(m.bias, 0)
            nn.init.constant_(m.weight, 1.0)

    def forward(self, x: torch.Tensor,
                return_attention: bool = False) -> Tuple[torch.Tensor, Optional[List[torch.Tensor]]]:
        """Forward pass.

        Args:
            x: Input images of shape (B, C, H, W)
            return_attention: Whether to return attention weights

        Returns:
            Class logits and optionally attention weights from all layers
        """
        # Patch embedding
        x = self.patch_embed(x)  # (B, N, embed_dim)

        # Add [CLS] token
        x = self.cls_token(x)  # (B, N+1, embed_dim)

        # Add positional embedding
        x = self.pos_embed(x)  # (B, N+1, embed_dim)

        # Apply transformer blocks
        attention_weights = []
        for block in self.blocks:
            x, attn = block(x, return_attention)
            if return_attention:
                attention_weights.append(attn)

        # Final layer norm
        x = self.norm(x)

        # Classification (only use [CLS] token)
        cls_output = x[:, 0]  # (B, embed_dim)
        logits = self.head(cls_output)  # (B, num_classes)

        if return_attention:
            return logits, attention_weights
        return logits, None

# Create ViT model
vit_model = VisionTransformer(
    img_size=224,
    patch_size=16,
    num_classes=1000,
    embed_dim=768,
    depth=12,
    num_heads=12
)

# Test the complete model
sample_batch = torch.randn(2, 3, 224, 224)
logits, attention_maps = vit_model(sample_batch, return_attention=True)

print(f"🎯 Complete ViT Model:")
print(f"Input shape: {sample_batch.shape}")
print(f"Output logits shape: {logits.shape}")
print(f"Number of attention maps: {len(attention_maps)}")
print(f"Each attention map shape: {attention_maps[0].shape}")

# Model statistics
total_params = sum(p.numel() for p in vit_model.parameters())
trainable_params = sum(p.numel() for p in vit_model.parameters() if p.requires_grad)

print(f"\n📊 Model Statistics:")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Model size: ~{total_params * 4 / 1024**2:.1f} MB (FP32)")

🎯 Complete ViT Model:
Input shape: torch.Size([2, 3, 224, 224])
Output logits shape: torch.Size([2, 1000])
Number of attention maps: 12
Each attention map shape: torch.Size([2, 12, 197, 197])

📊 Model Statistics:
Total parameters: 86,541,544
Trainable parameters: 86,541,544
Model size: ~330.1 MB (FP32)


## 🎓 **Summary**

In this notebook, we've built a complete Vision Transformer from scratch:

### **Key Components:**
1. **🔍 Patch Embedding**: Convert image patches to tokens
2. **🎯 [CLS] Token**: Aggregate representation for classification
3. **📍 Positional Encoding**: Provide spatial information
4. **🧠 Multi-Head Self-Attention**: Allow patches to attend to each other
5. **🔗 MLP**: Position-wise feed-forward network
6. **🏗️ Transformer Block**: Combine attention and MLP with residuals

### **Architecture Insights:**
- **Global attention** from the first layer
- **Residual connections** for stable training
- **Layer normalization** before operations (Pre-norm)
- **[CLS] token** as learnable aggregate representation

---

## 🔗 **Next Steps**
In the next notebook, we'll try to do attention visualization for more understanding